# Lecture 8: Sensitivity Analysis

---

```{note}
In Lecture 7, we solved the MTC Chennai bus-service LP by enumerating all basic feasible solutions (BFS) in a table, and identified the optimal solution $(x_1^*, x_2^*) = (20, 20)$ with $z^* = 140$ (₹'000/day). That answer assumed the problem data — contribution margins and resource limits — were known with certainty. In practice, they never are: fuel prices fluctuate, driver-shift budgets are revised, and planners ask "what if?" before committing to a schedule. **Sensitivity Analysis** is the systematic method for answering exactly these questions — without re-solving the LP from scratch.
```

**Learning Objectives**

By the end of this notebook, you will be able to:
1. Extract the inverse basis matrix $\mathbf{B}^{-1}$ directly from the optimal BFS table developed in Lecture 7.
2. Compute shadow prices, reduced costs, resource ranges, and objective ranges using $\mathbf{B}^{-1}$.
3. Interpret sensitivity results to make informed managerial decisions about resource allocation and service planning in transportation contexts.

**Prerequisites**: LP Formulation (Lecture 3), Graphical Method (Lecture 4), Python SciPy Method (Lecture 6), Simplex Method / BFS Table (Lecture 7)

**Estimated time**: 90 minutes (including in-class exercises)

---

## Why Sensitivity Analysis?

Every LP solution rests on a set of parameters: the objective-function coefficients $\mathbf{c}$, the constraint matrix $\mathbf{A}$, and the right-hand-side (RHS) vector $\mathbf{b}$. In real transportation planning, these parameters carry uncertainty:

- The driver-shift budget is set by HR and finance — it can be renegotiated.
- Over what range of driver-shift availability does each additional shift-hour continue to yield the same gain — and when does that rate change?
- The fuel allocation is controlled by the depot manager — it may be expanded for an additional cost.
- The contribution margin of a bus service depends on fare revenue and fuel cost — both volatile.

A decision-maker who takes the LP solution at face value treats these numbers as gospel. A decision-maker who understands sensitivity analysis asks:

> If I secure 5 more driver-shift-hours, how much additional revenue does MTC gain?

> Over what range of driver-shift availability does each additional shift-hour continue to yield the same gain — and when does that rate change?

> Over what range of fuel budget does the current service mix remain optimal?

> By how much can the contribution margin of premium express service fall before we should stop running it at full strength?"

Sensitivity analysis answers all fours questions — and it does so entirely from the optimal BFS table we already computed in Lecture 7.

---

### Shadow Prices

The **shadow price** $\pi_i^*$ of constraint $i$ is the rate of change of the optimal objective value $z^*$ with respect to a unit increase in $b_i$, holding all other data fixed:

$$\pi_i^* = \frac{\partial z^*}{\partial b_i}$$

For an LP in standard form, the vector of shadow prices is:

$$\boldsymbol{\pi}^* = \mathbf{c}_B^\top \mathbf{B}^{-1}$$

### Reduced Costs

The **reduced cost** $\bar{c}_j$ of variable $j$ measures the net change in the objective per unit increase in $x_j$ from zero, accounting for the fact that increasing $x_j$ forces adjustments to the basic variables to maintain feasibility:

$$\bar{c}_j = c_j - \boldsymbol{\pi}^* [\mathbf{A} \mid \mathbf{I}]_j$$

where $[\mathbf{A} \mid \mathbf{I}]_j$ is the $j$-th column of the augmented matrix.

For basic variables, the reduced cost is always zero: $\bar{c}_j = 0$ for $j \in$ basis. This is a useful sanity check.

For non-basic variables (currently at zero), the reduced cost indicates:
- $\bar{c}_j < 0$: increasing $x_j$ from zero reduces $z$ — correctly excluded from the basis in a maximisation problem.
- $\bar{c}_j = 0$: the problem has alternative optima — another BFS achieves the same $z^*$.
- $\bar{c}_j > 0$ (maximisation): should not occur at optimality — would mean the simplex missed a better solution.

### Resource Ranging

While, shadow prices tell us the *rate* at which $z^*$ changes with $b_i$, they are valid only while the current basis remains optimal. If $b_i$ increases too much, the geometry of the feasible region shifts enough that a different basis becomes optimal — and the shadow price changes. Thus, **resource ranging** determines the range $[b_i^{\min}, b_i^{\max}]$ over which $\pi_i^*$ remains valid (i.e., the current basis stays both feasible and optimal).

When $b_i$ changes by $\Delta_i$, the new basic variable values are:

$$\mathbf{x}_B(\Delta_i) = \mathbf{x}_B^* + \Delta_i \, \mathbf{B}^{-1}_{i}$$

where $\mathbf{B}^{-1}_{i}$ is the $i$-th column of $\mathbf{B}^{-1}$.

The current basis remains feasible as long as all basic variable values stay non-negative:

$$\mathbf{x}_B^* + \Delta_i \, \mathbf{B}^{-1}_{i} \geq \mathbf{0}$$

### Objective Ranging

Consistent with resource ranging, **objective ranging** asks: how far can $c_j$ shift before the current optimal basis changes? 

The current basis is optimal as long as all reduced costs of non-basic variables remain $\leq 0$ (for maximisation). When $c_j$ changes by $\Delta c_j$, only the reduced costs of the non-basic variables involving $x_j$ are affected.

For a basic variable $x_j$ with coefficient $c_j + \Delta c_j$, the updated shadow prices become:

$$\boldsymbol{\pi}^*(\Delta c_j) = (c_j + \Delta c_j) \cdot (\mathbf{B}^{-1})_{j\cdot} + \sum_{k \neq j} c_k \cdot (\mathbf{B}^{-1})_{k\cdot}$$

The reduced costs of the non-basic variables $s_1$ and $s_2$ then become:

$$\bar{c}_{s_k}(\Delta c_j) = \bar{c}_{s_k} - \Delta c_j \cdot (\mathbf{B}^{-1}\mathbf{A}_{s_k})_j$$

The basis remains optimal as long as $\bar{c}_{s_k}(\Delta c_j) \leq 0$ for all non-basic $k$.

---

## In-Class Exercise

### Exercise 1

*Same as Lecture 7 In-Class Exercise 1*

The Chennai Metropolitan Transport Corporation (MTC) is evaluating two new express services on a feeder corridor:

| Service | Net contribution margin (₹'000/trip) |
|---------|----------------------------------------|
| $\text{E}_1$ — Premium Express | 4 |
| $\text{E}_2$ — Limited-Stop Express | 3 |

Two resources constrain how many trips of each service MTC can run per day:

- Driver-shift hours: both services require 1 driver-shift-hour per trip; 40 driver-shift-hours are available per day.
- Fuel allocation: $\text{E}_1$ (longer route) consumes 2 units of fuel per trip, $\text{E}_2$ consumes 1 unit; the daily fuel budget is 60 units.

Maximize net contribution.

**Decision variables**: $x_1$ = daily trips of $\text{E}_1$ (Premium Express), $x_2$ = daily trips of $\text{E}_2$ (Limited-Stop Express).

**Objective**:
$$\max_{x_1,x_2} \; z = 4x_1 + 3x_2$$

**Subject to**:
$$\begin{aligned}
x_1 + x_2 &\leq 40 \quad \text{(driver-shift hours)} \\
2x_1 + x_2 &\leq 60 \quad \text{(fuel budget)} \\
x_1, x_2 &\geq 0
\end{aligned}$$

**Standard form** (introducing slack variables $s_1, s_2 \geq 0$):
$$\begin{aligned}
x_1 + x_2 + s_1 &= 40 \\
2x_1 + x_2 + s_2 &= 60 \\
x_1, x_2, s_1, s_2 &\geq 0
\end{aligned}$$

**Optimal BFS table** (from Lecture 7):

| Non-Basic (= 0) | Basic | $(x_1, x_2)$ | $(s_1, s_2)$ | Feasible? | $z = 4x_1+3x_2$ |
|-----------------|-------|--------------|--------------|-----------|-----------------|
| $x_1, x_2$ | $s_1, s_2$ | $(0, 0)$ | $(40, 60)$ | Yes | $0$ |
| $x_2, s_2$ | $x_1, s_1$ | $(30, 0)$ | $(10, 0)$ | Yes | $120$ |
| $x_2, s_1$ | $x_1, s_2$ | $(40, 0)$ | $(0, -20)$ | **No** | — |
| $x_1, s_2$ | $x_2, s_1$ | $(0, 60)$ | $(-20, 0)$ | **No** | — |
| $x_1, s_1$ | $x_2, s_2$ | $(0, 40)$ | $(0, 20)$ | Yes | $120$ |
| $s_1, s_2$ | $x_1, x_2$ | $(20, 20)$ | $(0, 0)$ | Yes | $\mathbf{140}$ |

At the optimal BFS, the **basic variables** are $x_1$ and $x_2$ (the non-basics $s_1, s_2$ are set to zero). The **basis matrix** $\mathbf{B}$ is formed by taking the columns of the augmented constraint matrix $[\mathbf{A} \mid \mathbf{I}]$ that correspond to the basic variables.

$$\mathbf{B} = \begin{bmatrix} 1 & 1 \\ 2 & 1 \end{bmatrix}, \qquad \mathbf{B}^{-1} = \frac{1}{\det(\mathbf{B})} \begin{bmatrix} 1 & -1 \\ -2 & 1 \end{bmatrix} = \begin{bmatrix} -1 & 1 \\ 2 & -1 \end{bmatrix}$$

since $\det(\mathbf{B}) = (1)(1) - (1)(2) = -1$.

Now then, everything about the optimal solution — and how it changes when parameters shift — can be read off from $\mathbf{B}^{-1}$. Specifically:

| Quantity | Formula | Interpretation |
|----------|---------|----------------|
| Shadow prices | $\boldsymbol{\pi}^* = \mathbf{c}_B^\top \mathbf{B}^{-1}$ | Value of relaxing each constraint by one unit |
| Reduced costs | $\bar{c}_j = c_j - \boldsymbol{\pi}^* \mathbf{A}_j$ | Net cost/benefit of forcing a non-basic variable into the basis |
| Resource ranging | $\mathbf{B}^{-1}(\mathbf{b} + \Delta\mathbf{b}) \geq \mathbf{0}$ | Range over which current basis remains feasible |
| Objective ranging | Track when $\bar{c}_j$ changes sign | Range over which current basis remains optimal |

We will now compute each of these systematically.

---

#### Shadow Price

$$\boldsymbol{\pi}^* = \mathbf{c}_B^\top \mathbf{B}^{-1} = \begin{bmatrix} 4 & 3 \end{bmatrix} \begin{bmatrix} -1 & 1 \\ 2 & -1 \end{bmatrix} = \begin{bmatrix} (-4+6) & (4-3) \end{bmatrix} = \begin{bmatrix} 2 & 1 \end{bmatrix}$$

So $\pi_1^* = 2$ and $\pi_2^* = 1$.

#### Managerial Interpretation

> **Shadow price $\pi_1^* = 2$ (driver-shift hours)**: If MTC could secure one additional driver-shift-hour per day (raising $b_1$ from 40 to 41), the daily contribution margin would increase by **₹2,000**. The shadow price tells the MTC planner exactly how much it is worth to negotiate with HR for an extra shift.

> **Shadow price $\pi_2^* = 1$ (fuel budget)**: One additional unit of fuel per day is worth ₹1,000 to MTC. If a fuel unit costs less than ₹1,000 to procure from an emergency supplier, it is profitable to acquire it.


### Reduced Cost

For the slack variables $s_1$ and $s_2$, the corresponding columns in the augmented matrix $[\mathbf{A} \mid \mathbf{I}]$ are the identity columns, with objective coefficients $c_{s_1} = c_{s_2} = 0$.

$$\bar{c}_{s_1} = 0 - \boldsymbol{\pi}^* [\mathbf{A} \mid \mathbf{I}]_1 = 0 - \begin{bmatrix}2 & 1\end{bmatrix}\begin{bmatrix}1\\0\end{bmatrix} = -2$$

$$\bar{c}_{s_2} = 0 - \boldsymbol{\pi}^* [\mathbf{A} \mid \mathbf{I}]_2 = 0 - \begin{bmatrix}2 & 1\end{bmatrix}\begin{bmatrix}0\\1\end{bmatrix} = -1$$

For the basic variables $x_1$ and $x_2$ (sanity check):

$$\bar{c}_{x_1} = 4 - \begin{bmatrix}2 & 1\end{bmatrix}\begin{bmatrix}1\\2\end{bmatrix} = 4 - (2+2) = 0 \ \checkmark$$

$$\bar{c}_{x_2} = 3 - \begin{bmatrix}2 & 1\end{bmatrix}\begin{bmatrix}1\\1\end{bmatrix} = 3 - (2+1) = 0 \ \checkmark$$

#### Managerial Interpretation

> **Reduced cost $\bar{c}_{s_1} = -2$**: Each unit of driver-shift slack (each idle driver-shift-hour) costs MTC ₹2,000 in foregone contribution. Equivalently, MTC is giving up ₹2,000 for every driver-shift-hour that sits unused. This mirrors $\pi_1^* = 2$ — shadow price and reduced cost of the corresponding slack are equal in magnitude but opposite in sign.

> **Reduced cost $\bar{c}_{s_2} = -1$**: Each unused fuel unit costs ₹1,000 in foregone contribution.

### Resource ranging

#### Resource ranging for $b_1$ (driver-shift hours, currently 40)

The first column of $\mathbf{B}^{-1}$ is $\begin{bmatrix}-1\\2\end{bmatrix}$, so:

$$\begin{bmatrix} x_1^*(\Delta_1) \\ x_2^*(\Delta_1) \end{bmatrix} = \begin{bmatrix}20\\20\end{bmatrix} + \Delta_1\begin{bmatrix}-1\\2\end{bmatrix} = \begin{bmatrix}20-\Delta_1\\20+2\Delta_1\end{bmatrix}$$

Requiring both $\geq 0$:
- $20 - \Delta_1 \geq 0 \Rightarrow \Delta_1 \leq 20$
- $20 + 2\Delta_1 \geq 0 \Rightarrow \Delta_1 \geq -10$

So $\Delta_1 \in [-10, 20]$, meaning $b_1 \in [30, 60]$.

> The shadow price $\pi_1^* = 2$ is valid for driver-shift budgets between 30 and 60 hours/day. Within this range, every additional shift-hour is worth exactly ₹2,000 to MTC.

#### Resource ranging for $b_2$ (fuel budget, currently 60)

The second column of $\mathbf{B}^{-1}$ is $\begin{bmatrix}1\\-1\end{bmatrix}$, so:

$$\begin{bmatrix} x_1^*(\Delta_2) \\ x_2^*(\Delta_2) \end{bmatrix} = \begin{bmatrix}20\\20\end{bmatrix} + \Delta_2\begin{bmatrix}1\\-1\end{bmatrix} = \begin{bmatrix}20+\Delta_2\\20-\Delta_2\end{bmatrix}$$

Requiring both $\geq 0$:
- $20 + \Delta_2 \geq 0 \Rightarrow \Delta_2 \geq -20$
- $20 - \Delta_2 \geq 0 \Rightarrow \Delta_2 \leq 20$

So $\Delta_2 \in [-20, 20]$, meaning $b_2 \in [40, 80]$.

> The shadow price $\pi_2^* = 1$ is valid for fuel budgets between 40 and 80 units/day. Each extra fuel unit yields exactly ₹1,000 additional contribution within this range.

### Cost Ranging

#### Ranging for $c_1$ (contribution margin of $\text{E}_1$, currently 4)

The non-basic variables are $s_1$ (column 3) and $s_2$ (column 4) of the sensitivity table:

- $\mathbf{B}^{-1}\mathbf{A}_{s_1} = \begin{bmatrix}-1\\2\end{bmatrix}$, with row 1 (for $x_1$) entry $= -1$.
- $\mathbf{B}^{-1}\mathbf{A}_{s_2} = \begin{bmatrix}1\\-1\end{bmatrix}$, with row 1 (for $x_1$) entry $= 1$.

Current reduced costs: $\bar{c}_{s_1} = -2$, $\bar{c}_{s_2} = -1$.

Updated reduced costs when $c_1 \to c_1 + \Delta c_1$:

$$\bar{c}_{s_1}(\Delta c_1) = -2 - \Delta c_1 \cdot (-1) = -2 + \Delta c_1 \leq 0 \Rightarrow \Delta c_1 \leq 2$$

$$\bar{c}_{s_2}(\Delta c_1) = -1 - \Delta c_1 \cdot (1) = -1 - \Delta c_1 \leq 0 \Rightarrow \Delta c_1 \geq -1$$

So $\Delta c_1 \in [-1, 2]$, meaning $c_1 \in [3, 6]$.

> The current service mix $(x_1^*, x_2^*) = (20, 20)$ remains optimal as long as $\text{E}_1$'s contribution margin stays between ₹3,000 and ₹6,000 per trip. If $c_1$ falls below ₹3,000, MTC should shift more trips to $\text{E}_2$.

#### Ranging for $c_2$ (contribution margin of $\text{E}_2$, currently 3)

Similarly, row 2 (for $x_2$) entries are: $\mathbf{B}^{-1}\mathbf{A}_{s_1}$ row 2 $= 2$; $\mathbf{B}^{-1}\mathbf{A}_{s_2}$ row 2 $= -1$.

$$\bar{c}_{s_1}(\Delta c_2) = -2 - \Delta c_2 \cdot 2 \leq 0 \Rightarrow \Delta c_2 \geq -1$$

$$\bar{c}_{s_2}(\Delta c_2) = -1 - \Delta c_2 \cdot (-1) = -1 + \Delta c_2 \leq 0 \Rightarrow \Delta c_2 \leq 1$$

So $\Delta c_2 \in [-1, 1]$, meaning $c_2 \in [2, 4]$.

> The current service mix remains optimal as long as $\text{E}_2$'s margin stays between ₹2,000 and ₹4,000 per trip. If $c_2$ reaches ₹4,000, the two services become equally attractive and MTC faces alternative optima.

---

## Take-Away Exercise

For the following take-away exercises, find the optimal solution, identify the basis, compute shadow prices, reduced costs, resource range, and objective range.

### Exercise 1

*Same as Lecture 7 Take-Away Exercise 2*

A Mumbai-based logistics firm dispatches two container types from its port yard each day: 20-ft containers ($x_1$) and 40-ft containers ($x_2$), earning net margins of ₹8,000 and ₹5,000 per dispatch (in ₹'000: $8$ and $5$).

Two yard resources constrain daily dispatches:

- Crane-hours: 4 hours/dispatch for 20-ft, 2 hours/dispatch for 40-ft; 60 crane-hours/day available.
- Gate-clearance hours: 2 hours/dispatch for 20-ft, 4 hours/dispatch for 40-ft; 48 gate-clearance-hours/day available.

Maximize net margins.

### Exercise 2

A Jaipur Outer Ring Road (ORR) operations centre manages daily tolling and maintenance crew allocation across two shift types:

- $x_1$: Day-shift crews (06:00–18:00) — net operational value ₹6,000/crew-day ($c_1 = 6$)
- $x_2$: Night-shift crews (18:00–06:00) — net operational value ₹4,000/crew-day ($c_2 = 4$)

Two resource constraints apply each day:
- Supervisor availability: 2 supervisors/day-crew + 1 supervisor/night-crew ≤ 50 supervisors available
- Vehicle allocation: 3 vehicles/day-crew + 2 vehicles/night-crew ≤ 90 vehicles available

Minimize operational cost.

### Exercise 3

*Same as Lecture 7 Take-Away Exercise 1*

A Lucknow-based freight operator must decide how many vehicle-trips of three types — Type A ($x_1$), Type B ($x_2$), Type C ($x_3$) — to deploy today, earning net profits of ₹50, ₹40, and ₹30 (in ₹'00/trip) respectively.

Three resources constrain the day's operations:

- Fuel: 3, 1, 2 units/trip for Types A, B, C; budget = 24 units.
- Loading-dock hours: 1, 2, 1 hours/trip; available = 16 hours.
- Driver shifts: 1, 1, 1 shift/trip; available = 10 shifts.

Maximize net profits.

---

## Circling Back

- **Lecture 4 (Graphical Method)**: Shadow prices correspond geometrically to the slope of the objective function's movement along the optimal vertex. RHS ranging corresponds to how far a constraint line can shift before the optimal vertex moves to a different corner.
- **Lecture 6 (Python SciPy)**: `scipy.optimize.linprog` with `method='highs'` returns shadow prices directly via `res.ineqlin.marginals`. The manual $\mathbf{B}^{-1}$ computation in this lecture explains *where* those numbers come from.
- **Lecture 7 (Simplex Method)**: The optimal BFS table identifies the basis $\mathbf{B}$. Sensitivity analysis is the natural sequel — it extracts maximum managerial value from the same computational artefact.


## Moving Forward 

- **Lecture 9 (Duality)**: Shadow prices $\boldsymbol{\pi}^* = \mathbf{c}_B^\top \mathbf{B}^{-1}$ are exactly the optimal solution of the dual LP associated with the primal. Duality theory provides a deeper theoretical foundation for why shadow prices exist and what they represent, and leads to the dual simplex method and complementary slackness conditions.
- **Lecture 10 (Transportation Problem)**: The transportation LP has a special structure that allows shadow prices to be interpreted directly as node potentials (supply and demand prices), connecting sensitivity analysis to economic equilibrium concepts.

---

## Further Reading

- Vanderbei, R.J. (2014). *Linear Programming: Foundations and Extensions* — Chapter 5: Sensitivity and Parametric Analysis.
- Bertsimas, D. and Tsitsiklis, J.N. (1997). *Introduction to Linear Optimization* — Chapter 5: Sensitivity Analysis.
- Winston, W.L. (2004). *Operations Research: Applications and Algorithms* — Chapter 5: Sensitivity Analysis.
- SciPy documentation: `scipy.optimize.linprog` sensitivity outputs — [docs.scipy.org](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.linprog.html)